# Philippine government sentiment study — 2026

**Group:** replace with your actual group name before submission.

This notebook uses the supplied **Bluesky and Mastodon CSV snapshot**, runs offline, and generates the reproducible analysis outputs. Download the whole uncompressed repository; the notebook needs `src/`, the project configuration, and `deliverables/public_2026/` beside it. Install the project following `README.md`, select that Python environment in Jupyter, and choose **Run All**. No social-media account or API token is needed for this handoff.

Use `deliverables/public_2026/README.md` for the handoff contents and `docs/methodology.md` for the study design. The attached *Midterm Activity 1* supplies assignment requirements. Save a submission copy with the instructor's required `GROUP NAME.ipynb` filename.

**Remaining group work:** review political relevance and source coverage, independently annotate sentiment, interpret actual topic/proximity results, enable evaluation after annotation, and write the final discussion. Neither blank labels nor rule predictions are human ground truth.

**Interpretation boundary:** sentiment in the collected posts is not a population approval estimate or an objective measurement of government performance. Performance-related topics identify discussion; claims about actual performance need separate evidence.

## 1. Setup and reproducibility

Run the README installation commands once in the environment selected by this notebook. The notebook locates the repository from its root or any subdirectory, reads the fixed handoff CSVs, and writes student outputs under `data/notebook-run/`. Keep the supplied snapshot unchanged so group members can reproduce the same starting corpus.

This workflow executes preparation, analysis, annotation-template generation, and optional evaluation only. It does not execute collection or access checks. Existing derived CSVs are reused on a repeat run, and existing human annotations are always preserved. Set `FORCE_REBUILD = True` only to replace derived outputs intentionally after changing inputs or configuration.

Setup verifies the snapshot SHA-256 manifest before using the supplied files. Missing or changed files produce an actionable error; restore the supplied snapshot rather than editing it.

In [ ]:
import csv
import hashlib
import html
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

from IPython.display import HTML, display


def find_repository() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError(
        "Open this notebook from the repository root or its notebooks directory."
    )


REPO = find_repository()
SNAPSHOT = REPO / "deliverables" / "public_2026"
CONFIG = SNAPSHOT / "config.toml"
RAW_FILES = [SNAPSHOT / "raw" / f"{platform}.csv" for platform in ("bluesky", "mastodon")]
ANNOTATION_SOURCE = SNAPSHOT / "labels" / "annotation.csv"
DATA_DIR = REPO / "data" / "notebook-run"
PROCESSED = DATA_DIR / "processed" / "posts.csv"
AUDIT = DATA_DIR / "processed" / "preprocessing_audit.csv"
ANALYSIS_DIR = DATA_DIR / "analysis"
ANNOTATIONS = DATA_DIR / "labels" / "annotation.csv"
EVALUATION_DIR = DATA_DIR / "evaluation"

missing_inputs = [
    path
    for path in (CONFIG, ANNOTATION_SOURCE, SNAPSHOT / "file_manifest.csv", *RAW_FILES)
    if not path.is_file()
]
if missing_inputs:
    missing = ", ".join(str(path.relative_to(REPO)) for path in missing_inputs)
    raise FileNotFoundError(
        f"Handoff files are missing: {missing}. Download the full uncompressed repository."
    )
if DATA_DIR.resolve().is_relative_to(SNAPSHOT.resolve()):
    raise ValueError("Keep student output files outside the supplied snapshot directory.")

# Set True only after reviewing which derived outputs will be replaced.
# Existing raw collections and human annotations are always preserved here.
FORCE_REBUILD = False


def run_cli(*arguments: str) -> None:
    if not arguments or arguments[0] not in {"prepare", "analyze", "label", "evaluate"}:
        raise ValueError("The handoff notebook accepts offline analysis commands only.")
    command = [sys.executable, "-m", "ph_sentiment", *map(str, arguments)]
    result = subprocess.run(command, cwd=REPO, capture_output=True, text=True, check=False)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        detail = result.stderr.strip() or "No diagnostic was returned."
        raise RuntimeError(f"Pipeline command failed ({result.returncode}): {detail}")
    if result.stderr.strip():
        print(result.stderr.strip())


def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def preview_csv(path: Path, limit: int = 6, max_columns: int = 8) -> None:
    if not path.exists():
        print(f"Not generated: {path.relative_to(REPO)}")
        return
    rows = read_csv(path)
    print(f"{path.relative_to(REPO)}: {len(rows):,} rows")
    if not rows:
        print("No rows to preview.")
        return

    columns = list(rows[0])[:max_columns]
    headings = "".join(f'<th scope="col">{html.escape(name)}</th>' for name in columns)
    body = []
    for row in rows[:limit]:
        values = []
        for name in columns:
            value = str(row.get(name, ""))
            if len(value) > 180:
                value = value[:180] + "…"
            values.append(f"<td>{html.escape(value)}</td>")
        body.append("<tr>" + "".join(values) + "</tr>")
    caption = html.escape(
        f"{path.name}: first {min(limit, len(rows))} rows; first {len(columns)} columns"
    )
    display(
        HTML(
            f"<table><caption>{caption}</caption><thead><tr>{headings}</tr></thead>"
            f"<tbody>{''.join(body)}</tbody></table>"
        )
    )


def overwrite_arguments() -> list[str]:
    return ["--overwrite"] if FORCE_REBUILD else []


# Verify the immutable handoff files before using their contents.
for entry in read_csv(SNAPSHOT / "file_manifest.csv"):
    supplied_file = (SNAPSHOT / entry["path"]).resolve()
    if not supplied_file.is_relative_to(SNAPSHOT.resolve()) or not supplied_file.is_file():
        raise ValueError(f"Invalid or missing snapshot file: {entry['path']}")
    with supplied_file.open("rb") as handle:
        digest = hashlib.file_digest(handle, "sha256").hexdigest()
    if digest != entry["sha256"]:
        raise ValueError(
            f"Snapshot checksum mismatch: {entry['path']}. "
            "Restore the supplied file; edit working copies under data/notebook-run instead."
        )
print("Supplied snapshot checksums verified.")
print(f"Repository: {REPO}")
print(f"Python: {sys.version.split()[0]}")
print(f"Interpreter: {sys.executable}")
print(f"Research configuration: {CONFIG.relative_to(REPO)}")
print(f"Student output directory: {DATA_DIR.relative_to(REPO)}")
print("Offline handoff: no collection or access-check commands will be run.")

## 2. Review the supplied collection and provenance

The handoff contains public records returned by the documented Bluesky and Mastodon APIs, with their original raw CSVs, collection manifests, collection report, run summary, and a reproducible configuration. The study period starts on **2026-01-01** and ends before **2027-01-01**; the snapshot covers only posts available when collection ran. Read the actual report rather than assuming a complete year or balanced platform coverage.

The notebook uses only these two source CSVs. It performs no live requests; all group members therefore analyze the same collection. For a separate future collection, follow the README's optional `collect-public` command and keep that run distinct from this snapshot.

The activity requires **at least 200 cleaned records from different sources**. The notebook checks the observed row and source counts after preparation. Political relevance still requires manual review; query matches can include unrelated references, news reports, and ambiguous targets.

In [ ]:
for filename in ("collection_report.csv", "run_summary.csv"):
    preview_csv(SNAPSHOT / filename)

for source in RAW_FILES:
    print(f"Supplied source: {source.relative_to(REPO)} ({len(read_csv(source)):,} raw rows)")

## 3. Prepare and audit the CSV corpus

The preparation stage applies configured date rules, normalization, and duplicate handling to the two supplied raw CSVs. It regenerates the cleaned corpus and audit in `data/notebook-run/processed/`; the original collection remains unchanged. Compare these outputs with the supplied `deliverables/public_2026/processed/` snapshot when checking reproducibility.

Preparation does not establish political relevance automatically. Review retained records and the audit before drawing conclusions. If relevance review leads to exclusions, document their record IDs and reasons, use a separately versioned input/configuration, and confirm that at least 200 relevant cleaned records and both sources remain. Do not erase existing annotations when revising the corpus.

In [ ]:
# RAW_FILES is an explicit two-source list defined in setup, never a directory glob.
prepared_outputs = (PROCESSED, AUDIT)
if any(path.exists() for path in prepared_outputs) and not FORCE_REBUILD:
    if not all(path.is_file() for path in prepared_outputs):
        raise RuntimeError(
            "The student preparation is incomplete. Set FORCE_REBUILD = True and rerun "
            "preparation to regenerate posts and audit; human annotations remain preserved."
        )
    print("Using existing student prepared CSV. Rebuild if its inputs or configuration changed.")
else:
    arguments = ["prepare", "--config", str(CONFIG)]
    for source in RAW_FILES:
        arguments.extend(["--input", str(source)])
    arguments.extend(
        [
            "--output",
            str(PROCESSED),
            "--audit-output",
            str(AUDIT),
            *overwrite_arguments(),
        ]
    )
    run_cli(*arguments)

preview_csv(AUDIT)
preview_csv(PROCESSED)

In [ ]:
if PROCESSED.exists():
    prepared_rows = read_csv(PROCESSED)
    platform_counts = Counter(row.get("platform", "") for row in prepared_rows)
    print(f"Final prepared rows: {len(prepared_rows):,}")
    print(f"Actual source counts: {dict(sorted(platform_counts.items()))}")
    populated_sources = [name for name, count in platform_counts.items() if name and count]
    if len(prepared_rows) < 200 or len(populated_sources) < 2:
        print(
            "Pilot only: the activity requires at least 200 cleaned records from different sources."
        )
    else:
        print(
            "Count/source checks passed. Confirm relevance, time coverage, "
            "and independence by inspecting the data and audit."
        )
else:
    prepared_rows = []
    print("Quality checks skipped: prepare the real corpus first.")

### Record the quality review

Complete these notes after inspecting real data:

- Collection time, query vocabulary, endpoint coverage, and observed date interval:
- Retained counts by platform and reasons for exclusions:
- Relevance checks, multilingual coverage, and suspected near-duplicate content:
- Remaining sampling limitations:

Do not interpret failed or empty source requests as neutral sentiment.

## 4. Prepare independent human sentiment labels

The handoff provides `deliverables/public_2026/labels/annotation.csv` with a blank `gold_label` column and no model predictions. This cell makes a student working copy in `data/notebook-run/labels/annotation.csv`; existing working labels are always preserved. If the prepared record IDs differ from the supplied template, it generates a new blank template instead.

**Keep annotators blind to model predictions:** assign reviewers the annotation CSV and these definitions before they inspect analysis/evaluation outputs. Reviewers should not use `analyzed_documents.csv` or copy the rule model's predictions. Per-document model predictions are hidden in the notebook by default.

Define the annotation target as sentiment toward the Philippine national administration or its governance:

- **positive:** clear approval or praise of the target.
- **negative:** clear criticism or disapproval of the target.
- **neutral:** discussion or information without a clear stance toward the target.

Review political relevance before labeling. Sarcasm, quoted claims, mixed stance, and ambiguous targets need review or adjudication. Do not label unresolved or irrelevant cases neutral just to fill a row. If possible, have a second reviewer independently label a subset and report actual agreement. Never invent human labels.

After annotation, save UTF-8 CSV without changing record IDs. If the corpus changes, create a separately named annotation file and reconcile IDs; do not erase completed human work.

In [ ]:
if ANNOTATIONS.exists():
    print(f"Preserving human annotation file: {ANNOTATIONS.relative_to(REPO)}")
elif not prepared_rows:
    print("Annotation template skipped: prepare real data first.")
else:
    supplied_annotations = read_csv(ANNOTATION_SOURCE)
    prepared_ids = [row["record_id"] for row in prepared_rows]
    supplied_ids = [row["record_id"] for row in supplied_annotations]
    if prepared_ids == supplied_ids:
        ANNOTATIONS.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ANNOTATION_SOURCE, ANNOTATIONS)
        print(f"Blank handoff template copied to: {ANNOTATIONS.relative_to(REPO)}")
    else:
        run_cli(
            "label",
            "--config",
            str(CONFIG),
            "--input",
            str(PROCESSED),
            "--output",
            str(ANNOTATIONS),
        )

if ANNOTATIONS.exists():
    annotation_rows = read_csv(ANNOTATIONS)
    annotation_ids = [row["record_id"] for row in annotation_rows]
    expected_ids = [row["record_id"] for row in prepared_rows]
    if len(annotation_ids) != len(set(annotation_ids)) or set(annotation_ids) != set(expected_ids):
        print(
            "Annotation record IDs differ from the prepared corpus. Existing human work is "
            "preserved; reconcile IDs in a separately named working CSV before evaluation."
        )
    label_counts = Counter(row.get("gold_label", "").strip() for row in annotation_rows)
    print(f"Human-label counts, including blanks: {dict(sorted(label_counts.items()))}")
    print("Complete and adjudicate gold_label independently before enabling evaluation.")
else:
    annotation_rows = []

## 5. Fit three LDA topics and compute rule sentiment

The analyzer defaults to three LDA topics with a fixed seed. LDA uses word counts; document proximity uses normalized TF-IDF and token sets. The fixed lexical baseline predicts positive, negative, or neutral sentiment with English/Filipino/Taglish rules.

The sentiment score is heuristic, not confidence. Rule matches can represent general emotional tone rather than stance toward the administration. Use independent human labels to measure that error. Keep record types and publication purposes separate: institutional posts, news reports, and personal opinions express different kinds of content. The summary groups record types separately and uses Manila-local months.

This stage writes model predictions to a separate analysis CSV. Give human reviewers only the annotation template until their independent labels are complete. Set `SHOW_DOCUMENT_PREDICTIONS = True` only after independent labeling or for a separate analyst who is not creating the gold labels.

In [ ]:
SHOW_DOCUMENT_PREDICTIONS = False
analysis_outputs = tuple(
    ANALYSIS_DIR / filename
    for filename in (
        "analyzed_documents.csv",
        "topic_terms.csv",
        "topic_metrics.csv",
        "sentiment_summary.csv",
        "proximity_pairs.csv",
    )
)

if not prepared_rows:
    print("Analysis skipped: no retained prepared records are available.")
elif any(path.exists() for path in analysis_outputs) and not FORCE_REBUILD:
    if not all(path.is_file() for path in analysis_outputs):
        raise RuntimeError(
            "Student analysis output is incomplete. Set FORCE_REBUILD = True and rerun "
            "analysis to regenerate all five CSVs; human annotations remain preserved."
        )
    print("Using existing analysis CSVs. Rebuild if the prepared corpus changed.")
else:
    run_cli(
        "analyze",
        "--config",
        str(CONFIG),
        "--input",
        str(PROCESSED),
        "--output-dir",
        str(ANALYSIS_DIR),
        *overwrite_arguments(),
    )

for filename in (
    "topic_terms.csv",
    "topic_metrics.csv",
    "sentiment_summary.csv",
):
    preview_csv(ANALYSIS_DIR / filename)

if SHOW_DOCUMENT_PREDICTIONS:
    preview_csv(ANALYSIS_DIR / "analyzed_documents.csv", max_columns=12)
else:
    print("Per-document predictions saved separately; keep them hidden from annotators.")

### Topic interpretation and topic-model evaluation

Name each topic only after reading its top terms and representative posts. Check overlapping terms, poorly assigned documents, and whether three topics are actually interpretable.

`topic_metrics.csv` records document-presence NPMI coherence, top-term diversity, and explicitly **in-sample** perplexity. These are descriptive diagnostics; they do not establish topic validity or predictive generalization.

| Topic | Name supported by records | Representative record IDs | Interpretation and limitations |
| --- | --- | --- | --- |
| 0 | Complete after inspection | Complete after inspection | Complete after inspection |
| 1 | Complete after inspection | Complete after inspection | Complete after inspection |
| 2 | Complete after inspection | Complete after inspection | Complete after inspection |

Report values from the generated topic metrics, together with the human review. Never replace missing topic output with assumed results.

## 6. Examine cosine, Jaccard, and Euclidean proximity

Larger cosine and Jaccard similarity mean more similar wording under their respective representations. Smaller Euclidean distance means closer normalized TF-IDF vectors. Similar wording does not establish agreement, accuracy, shared authorship, or coordinated behavior.

The pair limit bounds computation, so the CSV may contain only selected pairs. Inspect `pair_status` and exclude empty-representation results from substantive interpretation.

In [ ]:
preview_csv(ANALYSIS_DIR / "proximity_pairs.csv", limit=10)

### Proximity interpretation

Select actual pairs from the CSV and inspect both underlying posts:

- Pair IDs and the three measured values:
- Shared and differing wording that explains the comparison:
- Whether text similarity corresponds to stance similarity in this example:
- Pair-status and sampling limitations:

Keep examples traceable to record IDs. Do not quote unnecessary personal information in the submission.

## 7. Evaluate: stratified 90/10 holdout and 10-fold development CV

Only independent human labels count as ground truth. The evaluator creates a stratified 90% development / 10% holdout split, then uses ten stratified folds within development. All three classes must be present, and each development class must support ten folds. If a class is too small, obtain and annotate more relevant data instead of changing valid labels.

The rule model is fixed and has no fitting step. Its cross-validation scores describe variation across development subsets. They do not imply learning on training folds. Keep holdout results separate from development-fold metrics.

Inspect development errors before a rule revision. After inspecting holdout errors to change the rules, obtain a fresh untouched holdout for a final assessment.

After saving the working annotation CSV, set `RUN_EVALUATION = True` and rerun the following cell. Blank or invalid labels cause a clear evaluation error; no replacement labels or artificial metrics are generated. The annotation file is reloaded each time so newly saved human edits are used.

In [ ]:
# Deliberately enable only after independent annotation and adjudication.
RUN_EVALUATION = False
evaluation_outputs = tuple(
    EVALUATION_DIR / filename
    for filename in (
        "evaluation_metrics.csv",
        "classification_report.csv",
        "confusion_matrix.csv",
        "split_assignments.csv",
        "evaluation_predictions.csv",
    )
)
# Reload on every execution so reviewers' newly saved CSV edits are used.
annotation_rows = read_csv(ANNOTATIONS) if ANNOTATIONS.exists() else []

if not RUN_EVALUATION:
    print("Evaluation disabled until independent human annotation is complete.")
elif not annotation_rows:
    print("Evaluation skipped: no human annotation CSV is available.")
elif len({row["record_id"] for row in annotation_rows}) != len(annotation_rows) or {
    row["record_id"] for row in annotation_rows
} != {row["record_id"] for row in prepared_rows}:
    raise ValueError(
        "Annotation record IDs do not match the prepared corpus. Preserve existing human "
        "work and reconcile IDs in the working annotation CSV before evaluation."
    )
elif any(path.exists() for path in evaluation_outputs) and not FORCE_REBUILD:
    if not all(path.is_file() for path in evaluation_outputs):
        raise RuntimeError(
            "Student evaluation output is incomplete. Set FORCE_REBUILD = True and rerun "
            "evaluation for the same independently labeled corpus."
        )
    print("Using existing evaluation CSVs. Rebuild only for a documented new run.")
else:
    run_cli(
        "evaluate",
        "--config",
        str(CONFIG),
        "--input",
        str(ANNOTATIONS),
        "--output-dir",
        str(EVALUATION_DIR),
        *overwrite_arguments(),
    )

if RUN_EVALUATION:
    for filename in (
        "evaluation_metrics.csv",
        "classification_report.csv",
        "confusion_matrix.csv",
        "split_assignments.csv",
        "evaluation_predictions.csv",
    ):
        preview_csv(EVALUATION_DIR / filename)

### Evaluation interpretation

Complete these notes from the actual CSVs:

- Number of independently labeled posts, class balance, annotation process, and adjudication:
- Development-fold mean/variation and holdout accuracy, macro precision, recall, F1, and balanced accuracy:
- Per-class confusion and representative development errors:
- Model version, split reproducibility, and remaining language/target limitations:

Use scope fields to distinguish holdout results from development-fold results. A high score against labels copied from the model would be circular and invalid.

## 8. Scripts, outputs, and remarks

All tabular datasets generated by this workflow are CSV files. The supplied collection is ready to share; analysis and annotations from this notebook are written separately under `data/notebook-run/`.

| Script / command | Output | Remarks |
| --- | --- | --- |
| Supplied public API collection | `deliverables/public_2026/raw/bluesky.csv`, `raw/mastodon.csv` | Actual returned public records; manifests and collection reports document coverage. |
| `ph_sentiment prepare` | `data/notebook-run/processed/posts.csv`, `preprocessing_audit.csv` | Cleaned corpus and audit; check at least 200 relevant records and two sources. |
| `ph_sentiment label` or supplied template copy | `data/notebook-run/labels/annotation.csv` | Blank independent human annotation working copy; existing work is preserved. |
| `ph_sentiment analyze` | `analyzed_documents.csv`, `topic_terms.csv`, `topic_metrics.csv` | Three requested LDA topics, fixed-rule predictions, and topic diagnostics. |
| Same analysis command | `proximity_pairs.csv`, `sentiment_summary.csv` | Cosine, Jaccard, and Euclidean proximity, plus descriptive sentiment counts/shares. |
| `ph_sentiment evaluate` | Metrics, reports, confusion matrices, split assignments, predictions | Enabled only after human labeling; stratified 90/10 holdout and ten development folds. |
| This notebook | Saved executed cells, tables, and reviewed interpretation | Complete the discussion and rename a submission copy to `GROUP NAME.ipynb`. |

The group must finish the quality review, topic names and representative examples, proximity interpretations, independent labels, actual evaluation, scripts/output remarks, and final discussion. Keep outputs traceable to the supplied snapshot and record any corpus revisions.

Summarize sentiment by topic and source only within the observed corpus. Platform users self-select; APIs restrict coverage; duplicated content, bots, language, sarcasm, and target errors can distort results. Corpus sentiment alone does not measure actual government performance. Record these limits alongside any conclusion.